Generate independent candidate responses and select a consensus protocol under the configured call budget.


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import json
import hashlib
import re
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Any, List
import pandas as pd
import requests
from openai import OpenAI
from tqdm.auto import tqdm


In [ ]:
FOLDER_ID = ''
API_KEY = ''
LLM_API_URL = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'
INPUT_CSV = 'llmfmed_dataset.csv'
RAW_OUTPUT_CSV = 'usc_gpt_raw_drafts.csv'
OUTPUT_CSV_BY_K = {3: 'usc_gpt_sc3.csv', 5: 'usc_gpt_sc5.csv', 7: 'usc_gpt_sc7.csv'}
SC_MODEL_NAME = 'gpt120b_default'
SC_MODEL_URI = f'gpt://{FOLDER_ID}/gpt-oss-120b/latest'
CASE_COMPONENT_COLUMNS = ['Vignette', 'Review of Systems', 'Past Medical History', 'Physical Examination', 'Test Results']
EMERGENCY_LABELS = ['Self-care appropriate', 'Requires non-emergent care', 'Requires emergent care']
SUBCLINICAL_LABELS = ['Cardiology', 'Dermatology', 'Emergency', 'Endocrinology', 'Gastroenterology', 'General Medicine', 'General Surgery', 'Infectious Disease', 'Nephrology', 'Neurology', 'Obstetrics and Gynecology', 'Ophthalmology', 'Orthopedics', 'Other departments', 'Otolaryngology (ENT)', 'Pediatrics', 'Psychiatry', 'Pulmonology', 'Urology']
REQUEST_TIMEOUT = 120
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
K_SAMPLES = 7
K_LEVELS = [3, 5, 7]
TEMPERATURE_SC = 0.3
TEMPERATURE_USC = 0.3
MAX_TOKENS_PROTOCOL = 3000
MAX_TOKENS_USC = 200
CASE_CONCURRENCY = 5
SC_ROLE_NAME = 'Default'
SC_ROLE_DESCRIPTION = 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'

def detect_provider(model_uri: str) -> str:
    if 'aliceai-llm' in model_uri:
        return 'yandex_rest'
    return 'openai_compatible'
SC_PROVIDER = detect_provider(SC_MODEL_URI)
df = pd.read_csv(INPUT_CSV, sep=None, engine='python')
df = df.copy().reset_index(drop=True)
missing_case_columns = [col for col in CASE_COMPONENT_COLUMNS if col not in df.columns]
if missing_case_columns:
    raise ValueError(f'Missing case columns in dataset: {missing_case_columns}')
if max(K_LEVELS) > K_SAMPLES:
    raise ValueError('K_LEVELS не может превышать K_SAMPLES.')

def build_case_text(row_data: Dict[str, Any]) -> str:
    parts: List[str] = []
    for column in CASE_COMPONENT_COLUMNS:
        value = row_data.get(column)
        if pd.isna(value):
            continue
        value_str = str(value).strip()
        if not value_str or value_str.lower() == 'nan':
            continue
        parts.append(f'{column}:\n{value_str}')
    return '\n\n'.join(parts)
df['case_text'] = df.apply(lambda row: build_case_text(row.to_dict()), axis=1)
print('Rows:', len(df))
print('Model:', SC_MODEL_NAME, '->', SC_MODEL_URI)
print('Provider:', SC_PROVIDER)
print('K_SAMPLES:', K_SAMPLES, '| K_LEVELS:', K_LEVELS)
print('Outputs:', list(OUTPUT_CSV_BY_K.values()))


In [ ]:
session = requests.Session()
llm_cache: Dict[str, Dict[str, Any]] = {}
openai_client = OpenAI(api_key=API_KEY, base_url='https://llm.api.cloud.yandex.net/v1', project=FOLDER_ID)

def make_cache_key(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, sample_idx: int=0) -> str:
    payload = {'provider': provider, 'model_uri': model_uri, 'system_text': system_text, 'user_text': user_text, 'temperature': temperature, 'max_tokens': max_tokens, 'sample_idx': sample_idx}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _build_token_dict(prompt_tokens: int=0, completion_tokens: int=0, total_tokens: int=0) -> Dict[str, int]:
    return {'prompt_tokens': int(prompt_tokens or 0), 'completion_tokens': int(completion_tokens or 0), 'total_tokens': int(total_tokens or 0)}

def _extract_openai_text(response: Any) -> str:
    if not getattr(response, 'choices', None):
        return ''
    message = response.choices[0].message
    content = getattr(message, 'content', '')
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
            elif hasattr(item, 'type') and getattr(item, 'type', None) == 'text':
                parts.append(getattr(item, 'text', ''))
        return '\n'.join([p for p in parts if p]).strip()
    return ''

def _extract_openai_usage(usage: Any) -> Dict[str, int]:
    if usage is None:
        return _build_token_dict()
    prompt_tokens = getattr(usage, 'prompt_tokens', None)
    completion_tokens = getattr(usage, 'completion_tokens', None)
    total_tokens = getattr(usage, 'total_tokens', None)
    if total_tokens is None:
        input_tokens = getattr(usage, 'input_tokens', None)
        output_tokens = getattr(usage, 'output_tokens', None)
        prompt_tokens = prompt_tokens if prompt_tokens is not None else input_tokens
        completion_tokens = completion_tokens if completion_tokens is not None else output_tokens
        total_tokens = int(prompt_tokens or 0) + int(completion_tokens or 0)
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def _extract_yandex_usage(result: Dict[str, Any]) -> Dict[str, int]:
    usage = result.get('usage', {}) or {}
    prompt_tokens = usage.get('inputTextTokens', 0) or 0
    completion_tokens = usage.get('completionTokens', 0) or 0
    total_tokens = usage.get('totalTokens', 0) or prompt_tokens + completion_tokens
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def call_yandex_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, timeout: int=REQUEST_TIMEOUT, retries: int=MAX_RETRIES, sample_idx: int=0) -> Dict[str, Any]:
    cache_key = make_cache_key('yandex_rest', model_uri, system_text, user_text, temperature, max_tokens, sample_idx)
    if cache_key in llm_cache:
        return {'text': llm_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    headers = {'Content-Type': 'application/json', 'Authorization': f'Api-Key {API_KEY}'}
    payload = {'modelUri': model_uri, 'completionOptions': {'stream': False, 'temperature': temperature, 'maxTokens': max_tokens}, 'messages': [{'role': 'system', 'text': system_text}, {'role': 'user', 'text': user_text}]}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = session.post(LLM_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.RequestException as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        if response.status_code == 200:
            data = response.json()
            result = data.get('result', {}) or {}
            raw_text = result.get('alternatives', [{}])[0].get('message', {}).get('text', '').strip()
            usage = _extract_yandex_usage(result)
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, **usage, 'cached': False}
        last_error_text = f'HTTP {response.status_code}: {response.text}'
        if response.status_code in (429, 500, 502, 503, 504):
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        raise RuntimeError(last_error_text)
    raise RuntimeError(f'Yandex REST call failed after {retries} retries. Last error: {last_error_text}')

def call_openai_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, retries: int=MAX_RETRIES, sample_idx: int=0) -> Dict[str, Any]:
    cache_key = make_cache_key('openai_compatible', model_uri, system_text, user_text, temperature, max_tokens, sample_idx)
    if cache_key in llm_cache:
        return {'text': llm_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = openai_client.chat.completions.create(model=model_uri, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens)
            raw_text = _extract_openai_text(response)
            usage = _extract_openai_usage(getattr(response, 'usage', None))
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, **usage, 'cached': False}
        except Exception as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
    raise RuntimeError(f'OpenAI-compatible call failed after {retries} retries. Last error: {last_error_text}')

def call_llm(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, sample_idx: int=0) -> Dict[str, Any]:
    if provider == 'yandex_rest':
        return call_yandex_llm(model_uri, system_text, user_text, temperature, max_tokens, sample_idx=sample_idx)
    if provider == 'openai_compatible':
        return call_openai_llm(model_uri, system_text, user_text, temperature, max_tokens, sample_idx=sample_idx)
    raise ValueError(f'Unknown provider: {provider}')


In [ ]:
def build_stage1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are a medical expert.\nAt this stage, you need to evaluate a specific clinical case and provide a final structured protocol based on the case.\n\nUse only information explicitly present in the case. If a detail is absent, write "Not specified"; if a field is unnecessary, write "None".\nReason carefully from the case. Do not invent symptoms or findings absent from the case.\n'.strip()

def build_stage1_user_prompt(case_text: str, author_id: str) -> str:
    emergency_list = json.dumps(EMERGENCY_LABELS)
    subclinical_list = json.dumps(SUBCLINICAL_LABELS)
    return f'Read the clinical case and write one medical protocol.\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nWrite the protocol in plain text. Same field labels and order:\n\nAge:\nSex:\nComplaint:\nAnalysis:\nRequired diagnostic equipment/tests:\nConclusion:\nEmergency type:\nSubclinical specialization:\nRecommendations:\nNotes:\nProtocol Author: {author_id}\n\nField instructions:\nAge: copy from the case if stated; otherwise write "Not specified".\nSex: copy from the case if stated; otherwise write "Not specified".\nComplaint: briefly summarize the presenting problem from the case.\nAnalysis: explain the clinical reasoning, the strongest supporting facts, and the main relevant differential considerations.\nRequired diagnostic equipment/tests: mention only studies or equipment that are actually needed next; if nothing additional is needed, write "None".\nConclusion: give a concise clinical synthesis on the case.\nEmergency type: based on your analysis of this case, emergency type must equal exactly one of {emergency_list}.\nSubclinical specialization: based on your analysis of this case, subclinical specialization must equal exactly one of {subclinical_list}.\nRecommendations: provide practical, case-grounded next steps or management advice. Explicitly address the additional considerations.\nNotes: include only essential caveats; otherwise write "None".\n'.strip()

def strip_author_section(protocol_text: str) -> str:
    if protocol_text is None:
        return ''
    text = str(protocol_text).strip()
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)
    return parts[0].rstrip()

def build_usc_system_prompt() -> str:
    return 'You are a careful expert assistant. You select among candidate responses. You never write a new response of your own.'.strip()

def build_usc_user_prompt(case_text: str, drafts: List[str]) -> str:
    blocks = '\n\n'.join((f'Response {i}:\n{d}' for i, d in enumerate(drafts, start=1)))
    return f'I have generated the following responses to the clinical case below.\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\n{blocks}\n\nEvaluate these responses and select the most consistent response based on majority consensus.\nDo not write a new response. Do not merge or combine responses. Only select one of them.\n\nReturn plain text only, in exactly this format and nothing else:\nMost consistent response: <number>\n'.strip()

def parse_usc_choice(text: str, k: int) -> Dict[str, Any]:
    m = re.search('most\\s+consistent\\s+response\\s*:\\s*(\\d+)', str(text), flags=re.I)
    if m is None:
        m = re.search('(\\d+)', str(text))
    if m is not None:
        idx = int(m.group(1))
        if 1 <= idx <= k:
            return {'index': idx, 'parse_ok': True}
    return {'index': 1, 'parse_ok': False}


In [ ]:
def run_usc_case(row_data: Dict[str, Any]) -> Dict[str, Any]:
    case_text = row_data['case_text']
    system_text = build_stage1_system_prompt(SC_ROLE_NAME, SC_ROLE_DESCRIPTION)
    user_text = build_stage1_user_prompt(case_text, SC_MODEL_NAME)

    def _draft(sample_idx: int) -> Dict[str, Any]:
        return call_llm(provider=SC_PROVIDER, model_uri=SC_MODEL_URI, system_text=system_text, user_text=user_text, temperature=TEMPERATURE_SC, max_tokens=MAX_TOKENS_PROTOCOL, sample_idx=sample_idx)
    with ThreadPoolExecutor(max_workers=K_SAMPLES) as pool:
        draft_responses = list(pool.map(_draft, range(K_SAMPLES)))
    drafts = [r['text'].strip() for r in draft_responses]
    for i, d in enumerate(drafts, start=1):
        row_data[f'sc_draft_{i}'] = d
    for i, r in enumerate(draft_responses, start=1):
        row_data[f'draft_{i}_total_tokens'] = int(r['total_tokens'])

    def _usc(k: int) -> Dict[str, Any]:
        subset = drafts[:k]
        resp = call_llm(provider=SC_PROVIDER, model_uri=SC_MODEL_URI, system_text=build_usc_system_prompt(), user_text=build_usc_user_prompt(case_text, subset), temperature=TEMPERATURE_USC, max_tokens=MAX_TOKENS_USC)
        choice = parse_usc_choice(resp['text'], k)
        return {'k': k, 'resp': resp, 'choice': choice, 'winner': subset[choice['index'] - 1]}
    with ThreadPoolExecutor(max_workers=len(K_LEVELS)) as pool:
        usc_results = list(pool.map(_usc, K_LEVELS))
    for u in usc_results:
        k = u['k']
        d_prompt = sum((int(r['prompt_tokens']) for r in draft_responses[:k]))
        d_completion = sum((int(r['completion_tokens']) for r in draft_responses[:k]))
        d_total = sum((int(r['total_tokens']) for r in draft_responses[:k]))
        row_data[f'sc{k}_winner_protocol'] = u['winner']
        row_data[f'sc{k}_usc_selected_index'] = u['choice']['index']
        row_data[f'sc{k}_usc_parse_ok'] = u['choice']['parse_ok']
        row_data[f'sc{k}_usc_raw_response'] = u['resp']['text']
        row_data[f'sc{k}_draft_total_tokens'] = d_total
        row_data[f'sc{k}_usc_total_tokens'] = int(u['resp']['total_tokens'])
        row_data[f'sc{k}_prompt_tokens'] = d_prompt + int(u['resp']['prompt_tokens'])
        row_data[f'sc{k}_completion_tokens'] = d_completion + int(u['resp']['completion_tokens'])
        row_data[f'sc{k}_total_tokens'] = d_total + int(u['resp']['total_tokens'])
        row_data[f'sc{k}_cached'] = bool(u['resp']['cached']) and all((r['cached'] for r in draft_responses[:k]))
    return row_data

def run_usc_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    row_dicts = [row.to_dict() for _, row in input_df.iterrows()]
    with ThreadPoolExecutor(max_workers=CASE_CONCURRENCY) as pool:
        rows = list(tqdm(pool.map(run_usc_case, row_dicts), total=len(row_dicts), desc='SC@k + USC pipeline'))
    return pd.DataFrame(rows)
result_df = run_usc_pipeline(df)
print(result_df.shape)
for k in K_LEVELS:
    fails = int((~result_df[f'sc{k}_usc_parse_ok']).sum())
    mean_tok = result_df[f'sc{k}_total_tokens'].mean()
    print(f'SC@{k}: parse failures {fails}, mean tokens/case {mean_tok:,.0f}')
    print(result_df[f'sc{k}_usc_selected_index'].value_counts().sort_index().to_string())
result_df.head(3)


In [ ]:
DRAFT_COLUMNS = [f'sc_draft_{i}' for i in range(1, K_SAMPLES + 1)]
SERVICE_PREFIXES = tuple((f'sc{k}_' for k in K_LEVELS)) + ('draft_',)

def build_arm(result_df: pd.DataFrame, k: int) -> pd.DataFrame:
    keep = [c for c in result_df.columns if c not in DRAFT_COLUMNS and (not c.startswith(SERVICE_PREFIXES))]
    arm = result_df[keep].copy()
    arm['baseline_protocol'] = result_df[f'sc{k}_winner_protocol']
    arm['baseline_protocol_anonymous'] = arm['baseline_protocol'].map(strip_author_section)
    arm['baseline_model_id'] = SC_MODEL_NAME
    arm['baseline_model_uri'] = SC_MODEL_URI
    arm['baseline_provider'] = SC_PROVIDER
    arm['baseline_role_name'] = SC_ROLE_NAME
    arm['baseline_role_description'] = SC_ROLE_DESCRIPTION
    arm['sc_k'] = k
    arm['sc_temperature'] = TEMPERATURE_SC
    arm['usc_selected_index'] = result_df[f'sc{k}_usc_selected_index']
    arm['usc_parse_ok'] = result_df[f'sc{k}_usc_parse_ok']
    arm['usc_raw_response'] = result_df[f'sc{k}_usc_raw_response']
    arm['draft_total_tokens'] = result_df[f'sc{k}_draft_total_tokens']
    arm['usc_total_tokens'] = result_df[f'sc{k}_usc_total_tokens']
    arm['prompt_tokens'] = result_df[f'sc{k}_prompt_tokens']
    arm['completion_tokens'] = result_df[f'sc{k}_completion_tokens']
    arm['total_tokens'] = result_df[f'sc{k}_total_tokens']
    arm['cached'] = result_df[f'sc{k}_cached']
    return arm

def build_token_summary(arm: pd.DataFrame, k: int) -> pd.DataFrame:
    return pd.DataFrame([{'model_id': SC_MODEL_NAME, 'model_uri': SC_MODEL_URI, 'provider': SC_PROVIDER, 'role_name': SC_ROLE_NAME, 'sc_k': k, 'sc_temperature': TEMPERATURE_SC, 'draft_total_tokens': int(arm['draft_total_tokens'].fillna(0).sum()), 'usc_total_tokens': int(arm['usc_total_tokens'].fillna(0).sum()), 'prompt_tokens': int(arm['prompt_tokens'].fillna(0).sum()), 'completion_tokens': int(arm['completion_tokens'].fillna(0).sum()), 'total_tokens': int(arm['total_tokens'].fillna(0).sum()), 'mean_total_tokens_per_case': float(arm['total_tokens'].fillna(0).mean()), 'n_cases': len(arm), 'n_cached': int(arm['cached'].fillna(False).sum()), 'n_usc_parse_fail': int((~arm['usc_parse_ok'].fillna(False)).sum())}])
arms = {k: build_arm(result_df, k) for k in K_LEVELS}
summaries = {k: build_token_summary(arms[k], k) for k in K_LEVELS}
pd.concat(list(summaries.values()), ignore_index=True)


In [ ]:
result_df.to_csv(RAW_OUTPUT_CSV, sep=';', index=False, encoding='utf-8-sig')
print('Saved:', RAW_OUTPUT_CSV, result_df.shape)
for k in K_LEVELS:
    arm_part = arms[k].copy()
    arm_part['table_source'] = 'result_df'
    summary_part = summaries[k].copy()
    summary_part['table_source'] = 'token_summary_df'
    combined_output_df = pd.concat([arm_part, summary_part], ignore_index=True, sort=False)
    out_csv = OUTPUT_CSV_BY_K[k]
    combined_output_df.to_csv(out_csv, sep=';', index=False, encoding='utf-8-sig')
    print('Saved:', out_csv, '| rows:', len(arm_part), '| total tokens:', summaries[k]['total_tokens'].iloc[0], '| mean/case:', round(summaries[k]['mean_total_tokens_per_case'].iloc[0], 1))
identical = (result_df[DRAFT_COLUMNS].nunique(axis=1) == 1).sum()
print(f'\nКейсов, где все {K_SAMPLES} черновиков идентичны: {identical} из {len(result_df)}' + ('  <-- ПРОВЕРИТЬ КЭШ И ТЕМПЕРАТУРУ' if identical else '  (ок)'))
